In [ ]:
#| default_exp phone

# The whole voice, off PyTorch

> Kokoro's front half and sine source converted for both phones at any length, the decoders shrunk for download, and
> the glue an app needs, scored against Kokoro on the Mac.

Notebooks 103 and 104 put the decoder on each phone. A phone needs the rest too: everything `analyse` does after
turning text into phonemes (the next notebook's job), and the decoder's sine source, which stayed in PyTorch. This
notebook converts those, makes everything small enough to offer as a download, and then runs the whole pipeline the
way an app will: each platform's runtimes, and only the glue code an app would have to write.

It converts models and needs Google's converter, so `nbdev-test` skips it.

In [ ]:
#| export
import copy
import os
import shutil
import subprocess
import sys
import time
import warnings

import coremltools as ct
import coremltools.optimize.coreml as cto
import numpy as np
import torch
import torch.nn.functional as F

from text_to_speech.core import DATA
from text_to_speech.coreml import BUCKETS, COREML, HarmonicSource, compile_coreml, exportable_decoder, tile
from text_to_speech.litert import phone_decoder
from text_to_speech.voice import FRAME, SR, analyse, decode, model, pipeline

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.


Torch version 2.15.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.8.0 is the most recent version that has been tested.


In [ ]:
import librosa
from fastcore.test import test_close, test_eq

from text_to_speech.data import metadata

Everything this notebook makes goes under `data/voice`, one folder per platform: what each app would download.
Two crashes shape how it runs. ONNX Runtime and LiteRT, Android's runtimes, crash when they share a process with
Core ML here, so anything that runs them runs `isolated`, in a Python of its own. And converting a model after
predicting with another crashed Core ML's Python bindings, so conversions run `fresh`, from this module's exported
functions. The second turned out to be more general (it's what notebook 103 met too): Core ML lets go of a
prediction's inputs a little later, on a thread of its own, and if that frees a numpy array, Python crashes. So every
prediction goes through `predict`, which keeps its inputs alive for a minute.

In [ ]:
#| export
VOICE = DATA / "voice"
IOS, ANDROID = VOICE / "ios", VOICE / "android"


def isolated(code, prelude="import numpy as np", **arrays):
    "Run `code` in a fresh Python, with `arrays` as the dict `inputs`; it puts arrays in the dict `outputs`."
    folder = VOICE / "parts"
    folder.mkdir(parents=True, exist_ok=True)
    np.savez(folder / "in.npz", **arrays)
    script = (
        f"{prelude}\nimport numpy as np\ninputs = dict(np.load({str(folder / 'in.npz')!r}))\noutputs = {{}}\n"
        f"{code}\nnp.savez({str(folder / 'out.npz')!r}, **outputs)"
    )
    r = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, check=False)
    if r.returncode:
        raise RuntimeError(r.stderr[-3000:])
    return dict(np.load(folder / "out.npz"))


def fresh(code, **arrays):
    "`isolated`, with this module imported: for converting models."
    prelude = "import torch, coremltools as ct, coremltools.optimize.coreml as cto\nfrom text_to_speech.phone import *"
    return isolated(code, prelude=prelude, **arrays)


_feeds = []  # (when, inputs) of recent predictions


def predict(m, feed):
    "m.predict(feed), keeping feed alive for a minute: Core ML lets go of its inputs later, on a thread of its own."
    now = time.monotonic()
    _feeds[:] = [(t, f) for t, f in _feeds if now - t < 60] + [(now, feed)]
    return m.predict(feed)


def size_mb(path):
    "Size of a file, or of everything in a folder, in MB."
    path = str(path)
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e6
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6

## The bar

The same as appendix 193's: ten sentences, and the distance between log mel spectrograms, against the distance
between two of Kokoro's own runs with different seeds. Anything here should land where notebook 103's Core ML decoder
did, about 3.9 times that floor: most of that is the padding fixed sizes need, which costs nothing in predicted MOS.
Where a change moves the timing a little, the frame-by-frame distance punishes the shift more than any change in the
sound, so there's also the distance after aligning the two in time.
(UTMOS, appendix 193's MOS predictor, needs `torchaudio`, which has no Python 3.14 build yet, so this notebook goes by
distance alone.)

In [ ]:
texts = ["Speed comes from the voice itself, not from stretching the audio afterwards."]
texts += [t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9]
utterances = [analyse(t) for t in texts]


def mel(a):
    return np.log(np.maximum(librosa.feature.melspectrogram(y=a, sr=SR, n_fft=1024, hop_length=256, n_mels=80), 1e-5))


def distance(a, b):
    n = min(len(a), len(b))
    return np.abs(mel(a[:n]) - mel(b[:n])).mean()


def aligned(a, b):
    "The same distance after dynamic time warping: fair to audio whose timing differs a little."
    x, y = mel(a), mel(b)
    D, path = librosa.sequence.dtw(X=x, Y=y, metric="cityblock")
    return D[-1, -1] / (len(path) * x.shape[0])


def kokoro(seed):
    torch.manual_seed(seed)
    return [decode(u) for u in utterances]


ref, other = kokoro(0), kokoro(1)
seconds = sum(map(len, ref)) / SR
floor = np.mean([distance(a, b) for a, b in zip(ref, other)])
aligned_floor = np.mean([aligned(a, b) for a, b in zip(ref, other)])
print(f"{seconds:.1f}s of audio; two seeds differ by {floor:.3f}, or {aligned_floor:.3f} aligned")

61.8s of audio; two seeds differ by 0.070, or 0.070 aligned


## The front half as two modules

`analyse` runs Kokoro's ALBERT, its duration predictor and its text encoder over the phonemes, decides how many 25 ms
frames each phoneme lasts, then runs the pitch and energy predictors over the frames. The step in the middle, turning
durations into frames, is a loop over phonemes with a data-dependent size: no converter takes it, and it's two lines
of app code anyway. So the front half becomes two models with that step between them: `Tokens`, per phoneme, and
`Prosody`, per frame.

Kokoro's modules take masks and lengths so that they can run a batch of sentences of different lengths, and pack
their LSTMs' inputs to match. An app runs one sentence at a time, so the rewrite drops all of that, and folds weight
normalisation into the weights as notebook 103 did.

In [ ]:
#| export
def folded(m):
    "A copy of `m` with weight normalisation folded into its weights."
    for x in m.modules():
        if hasattr(x, "weight_g"):  # weight_norm caches a computed weight, which deepcopy refuses
            x.weight = x.weight.detach()
    m = copy.deepcopy(m).eval()
    for x in m.modules():
        if hasattr(x, "weight_g"):
            torch.nn.utils.remove_weight_norm(x)
    return m.requires_grad_(False)


class Tokens(torch.nn.Module):
    "Phoneme ids (1, n) and a voice's style (1, 256) -> features per phoneme d, t_en, and each one's duration in frames."

    def __init__(self):
        super().__init__()
        self.bert, self.bert_encoder = folded(model.bert), folded(model.bert_encoder)
        self.predictor, self.text_encoder = folded(model.predictor), folded(model.text_encoder)

    def forward(self, ids, style):
        s = style[:, 128:]
        x = self.bert_encoder(self.bert(ids, attention_mask=torch.ones_like(ids)))  # (1, n, 512)
        se = s[:, None].expand(1, x.shape[1], 128)
        for block in self.predictor.text_encoder.lstms:  # DurationEncoder.forward, for one unpadded sentence
            if isinstance(block, torch.nn.LSTM):
                x = block(torch.cat([x, se], -1) if x.shape[-1] == 512 else x)[0]
            else:  # AdaLayerNorm, then the style again
                h = block.fc(s)[:, None]
                x = torch.cat([F.layer_norm(x, (512,), eps=block.eps) * (1 + h[..., :512]) + h[..., 512:], se], -1)
        d = x  # (1, n, 640)
        duration = torch.sigmoid(self.predictor.duration_proj(self.predictor.lstm(d)[0])).sum(-1)
        t = self.text_encoder.embedding(ids).transpose(1, 2)
        for c in self.text_encoder.cnn:
            t = c(t)
        t_en = self.text_encoder.lstm(t.transpose(1, 2))[0].transpose(1, 2)  # (1, 512, n)
        return d, t_en, duration


class Prosody(torch.nn.Module):
    "Duration features repeated out to frames (1, 640, T) and the style -> pitch and energy, two values a frame."

    def __init__(self):
        super().__init__()
        self.predictor = folded(model.predictor)

    def forward(self, en, style):
        return self.predictor.F0Ntrain(en, style[:, 128:])

Between them, the app does what `analyse` does in the middle: rounds the durations to whole frames, at least one
each, and repeats every phoneme's features once per frame it lasts. (`np.round` rounds halves to even, as
`torch.round` does. An app has to round the same way, or now and then a phoneme comes out a frame longer.)

In [ ]:
#| export
def phoneme_ids(phonemes):
    "Kokoro's input for a phoneme string: its vocabulary's ids, with a silence at each end."
    return np.array([[0, *[model.vocab[p] for p in phonemes if p in model.vocab], 0]])


def voice_style(voice, phonemes):
    "A voice's style for a sentence: its pack holds one row per phoneme count."
    return pipeline(voice).load_voice(voice)[len(phonemes) - 1].numpy().astype(np.float32)


def frames(duration):
    "Each phoneme's frames at 1x: rounded half to even, and at least one."
    return np.maximum(1, np.round(np.asarray(duration).reshape(-1))).astype(int)


def per_frame(x, frames):
    "Repeat each phoneme's column of x (1, channels, n) once per frame it lasts."
    return np.ascontiguousarray(np.repeat(x, frames, axis=-1))


def repeat_to(x, n):
    "Notebook 103's padding, in numpy: repeat x along its last axis to length n."
    return np.concatenate([x] * -(-n // x.shape[-1]), -1)[..., :n]

In [ ]:
tokens, prosody = Tokens().eval(), Prosody().eval()
u = utterances[0]
ids, style = phoneme_ids(u.phonemes), voice_style(u.voice, u.phonemes)
with torch.no_grad():
    d, t_en, duration = tokens(torch.tensor(ids), torch.tensor(style))
    f = frames(duration)
    F0, N = prosody(torch.tensor(per_frame(d.transpose(1, 2).numpy(), f)), torch.tensor(style))
test_eq(f, u.frames.numpy())
test_eq(per_frame(t_en.numpy(), f), u.asr.numpy())
test_eq(F0.numpy(), u.F0.numpy())
test_eq(N.numpy(), u.N.numpy())

Exactly `analyse`'s frames, features, pitch and energy.

## Any length

Notebook 103 gave the decoder fixed sizes and padded sentences up to them. That won't do here: the LSTMs read in both
directions, so padding on the end changes what the backward direction carries into every real phoneme, and the
durations with it. These models need the sentence's own length. `torch.export` won't give them that:

In [ ]:
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="The tensor attributes")  # The LSTMs' weights, about to be unrolled
        torch.export.export(
            tokens, (torch.tensor(ids), torch.tensor(style)), dynamic_shapes=({1: torch.export.Dim("n")}, None)
        )
except torch._dynamo.exc.UserError as e:
    print(str(e).strip().splitlines()[0][:200])

Constraints violated (n)! For more information, run with TORCH_LOGS="+dynamic".


It unrolls each LSTM into one step per phoneme, at this sentence's length. Both runtimes have an LSTM of their own that
takes any length, though, and the older routes into them keep `torch.nn.LSTM` whole: for Core ML, tracing with
TorchScript (deprecated in PyTorch, still supported by coremltools), with flexible input shapes; for Android, the ONNX
exporter, run by ONNX Runtime. LiteRT, Android's route to the GPU for the decoder, has no such LSTM, and the front
half is a tenth of the work, so it can run on the CPU.

In [ ]:
#| export
def to_coreml_flexible(module, args, inputs, outputs, path, precision=ct.precision.FLOAT32):
    "Trace `module` at `args` and convert it with `inputs` (ct.TensorTypes, flexible shapes allowed); compile at `path`."
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*torch.jit")  # Deprecated, and the only route that keeps the LSTMs
        traced = torch.jit.trace(module, args)
    m = ct.convert(
        traced,
        inputs=inputs,
        outputs=[ct.TensorType(o) for o in outputs],
        compute_precision=precision,
        minimum_deployment_target=ct.target.iOS18,
    )
    remove_coreml(path)
    return compile_coreml(m, path)


def remove_coreml(path):
    "Delete an .mlpackage and its compiled .mlmodelc: compiling won't overwrite one."
    for p in [str(path), str(path).replace(".mlpackage", ".mlmodelc")]:
        shutil.rmtree(p, ignore_errors=True)


def to_onnx(module, args, path, inputs, outputs, dynamic):
    "Export `module` at `args` to ONNX at `path`, with the named axes in `dynamic` free to change size."
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        torch.onnx.export(
            module,
            args,
            str(path),
            input_names=inputs,
            output_names=outputs,
            dynamic_axes=dynamic,
            opset_version=17,
            dynamo=False,
        )

In [ ]:
#| export
TOKENS_IN = [
    ct.TensorType("ids", shape=(1, ct.RangeDim(3, 512, 64)), dtype=np.int32),
    ct.TensorType("style", shape=(1, 256)),
]
PROSODY_IN = [ct.TensorType("en", shape=(1, 640, ct.RangeDim(4, 4096, 256))), ct.TensorType("style", shape=(1, 256))]


def build_front(ids, style):
    "Convert Tokens and Prosody for both platforms, at the shapes of one sentence, into IOS and ANDROID."
    IOS.mkdir(parents=True, exist_ok=True)
    ANDROID.mkdir(parents=True, exist_ok=True)
    tokens, prosody = Tokens().eval(), Prosody().eval()
    ids, style = torch.tensor(ids), torch.tensor(style)
    with torch.no_grad():
        d, _, duration = tokens(ids, style)
        en = torch.tensor(per_frame(d.transpose(1, 2).numpy(), frames(duration)))
    to_coreml_flexible(tokens, (ids, style), TOKENS_IN, ["d", "t_en", "duration"], IOS / "tokens.mlpackage")
    to_coreml_flexible(prosody, (en, style), PROSODY_IN, ["F0", "N"], IOS / "prosody.mlpackage")
    to_onnx(
        tokens,
        (ids, style),
        ANDROID / "tokens.onnx",
        ["ids", "style"],
        ["d", "t_en", "duration"],
        {"ids": {1: "n"}, "d": {1: "n"}, "t_en": {2: "n"}, "duration": {1: "n"}},
    )
    to_onnx(
        prosody,
        (en, style),
        ANDROID / "prosody.onnx",
        ["en", "style"],
        ["F0", "N"],
        {"en": {2: "frames"}, "F0": {1: "frames2"}, "N": {1: "frames2"}},
    )

In [ ]:
fresh('build_front(inputs["ids"], inputs["style"])', ids=ids, style=style)
for name in ["tokens.mlpackage", "prosody.mlpackage"]:
    print(f"{name}: {size_mb(IOS / name):.0f} MB")
for name in ["tokens.onnx", "prosody.onnx"]:
    print(f"{name}: {size_mb(ANDROID / name):.0f} MB")

tokens.mlpackage: 78 MB
prosody.mlpackage: 34 MB
tokens.onnx: 78 MB
prosody.onnx: 34 MB


Checked on a sentence of a different length from the one they were converted at:

In [ ]:
u2 = utterances[3]
ids2, style2 = phoneme_ids(u2.phonemes), voice_style(u2.voice, u2.phonemes)
coreml_tokens = ct.models.CompiledMLModel(str(IOS / "tokens.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
coreml_prosody = ct.models.CompiledMLModel(str(IOS / "prosody.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
c = predict(coreml_tokens, {"ids": ids2.astype(np.int32), "style": style2})
en2 = per_frame(c["d"].transpose(0, 2, 1), frames(c["duration"]))
o = isolated(
    f"""
import onnxruntime as ort
tokens = ort.InferenceSession("{ANDROID / "tokens.onnx"}", providers=["CPUExecutionProvider"])
prosody = ort.InferenceSession("{ANDROID / "prosody.onnx"}", providers=["CPUExecutionProvider"])
outputs["duration"] = tokens.run(None, {{"ids": inputs["ids"], "style": inputs["style"]}})[2]
outputs["F0"] = prosody.run(None, {{"en": inputs["en"], "style": inputs["style"]}})[0]
""",
    ids=ids2,
    style=style2,
    en=en2,
)
test_eq(frames(c["duration"]), u2.frames.numpy())
test_eq(frames(o["duration"]), u2.frames.numpy())
for out, name in [(predict(coreml_prosody, {"en": en2, "style": style2})["F0"], "Core ML"), (o["F0"], "ONNX")]:
    print(f"{name}: pitch within {np.abs(out - u2.F0.numpy()).max():.4f} Hz of analyse's")
test_close(per_frame(c["t_en"], frames(c["duration"])), u2.asr.numpy(), eps=1e-3)

Core ML: pitch within 0.0027 Hz of analyse's
ONNX: pitch within 0.0027 Hz of analyse's


The same frames as `analyse`, and pitch to a ten-thousandth of a hertz. Both are 32-bit, and have to be: at 16 bits
the ALBERT inside `Tokens` overflows, and Core ML returns NaN for every duration:

In [ ]:
half = fresh(
    """
args = torch.tensor(inputs["ids"]), torch.tensor(inputs["style"])
half = to_coreml_flexible(Tokens().eval(), args, TOKENS_IN, ["d", "t_en", "duration"], VOICE / "tokens16.mlpackage",
                          precision=ct.precision.FLOAT16)
outputs["duration"] = predict(half, {"ids": inputs["ids2"].astype(np.int32), "style": inputs["style2"]})["duration"]
remove_coreml(VOICE / "tokens16.mlpackage")
""",
    ids=ids,
    style=style,
    ids2=ids2,
    style2=style2,
)
print(half["duration"][0, :8])

[nan nan nan nan nan nan nan nan]


Their weights can still be stored in 8 bits and computed in 32, which is a quarter of the download. We'll see what
that costs once the whole pipeline is together.

In [ ]:
#| export
LINEAR8 = cto.OptimizationConfig(global_config=cto.OpLinearQuantizerConfig(mode="linear_symmetric"))


def quantize_coreml(name):
    "IOS/name.mlpackage with its weights in 8 bits, as IOS/name8, compiled."
    m = cto.linear_quantize_weights(ct.models.MLModel(str(IOS / f"{name}.mlpackage"), skip_model_load=True), LINEAR8)
    remove_coreml(IOS / f"{name}8.mlpackage")
    m.save(str(IOS / f"{name}8.mlpackage"))
    ct.utils.compile_model(str(IOS / f"{name}8.mlpackage"), str(IOS / f"{name}8.mlmodelc"))

In [ ]:
for name in ["tokens", "prosody"]:
    fresh(f"quantize_coreml({name!r})")
    print(f"{name}: {size_mb(IOS / f'{name}.mlpackage'):.0f} MB -> {size_mb(IOS / f'{name}8.mlpackage'):.0f} MB")

tokens: 78 MB -> 20 MB


prosody: 34 MB -> 9 MB


## The sine source

Notebook 103 left `HarmonicSource` in PyTorch: on the Mac, a converted copy crashed PyTorch in the same process. A
phone has no PyTorch, but the source has two other problems. It draws random numbers, and it interpolates by a factor
of 1/300, which Core ML's converter refuses. So `Source` takes its random numbers as inputs, and does the two
interpolations as the arithmetic they amount to. Kokoro's pitch is constant over each 300-sample step, so the
downsampling is the mean of the middle two samples of each step, and upsampling blends each frame with its nearer
neighbour by a fixed weight.

In [ ]:
#| export
class Source(torch.nn.Module):
    "HarmonicSource, deterministic: the caller supplies the initial phases (the first zero) and unit Gaussian noise."

    def __init__(self, dec):
        super().__init__()
        self.gen = dec.generator
        sg = self.gen.m_source.l_sin_gen
        k = sg.upsample_scale
        w = (torch.arange(k) + 0.5) / k - 0.5  # Where each output sample falls between its frame and the next
        self.register_buffer("w", w.abs())
        self.register_buffer("before", (w < 0).float())
        self.register_buffer("harmonics", torch.arange(1, sg.harmonic_num + 2, dtype=torch.float32))

    def upsample(self, x):
        "F.interpolate(x, scale_factor=k, mode='linear'): each output a blend of its frame and the nearer neighbour."
        prev, nxt = torch.cat([x[..., :1], x[..., :-1]], -1), torch.cat([x[..., 1:], x[..., -1:]], -1)
        neighbour = prev[..., None] * self.before + nxt[..., None] * (1 - self.before)  # (1, 9, n, k)
        return (x[..., None] * (1 - self.w) + neighbour * self.w).reshape(x.shape[0], x.shape[1], -1)

    def forward(self, f0, phases, noise):
        "f0 (1, 2T), phases (1, 9), noise (1, 9, 600T) -> the STFT of the source, (1, 22, 120T + 1)."
        g, sg = self.gen, self.gen.m_source.l_sin_gen
        k = sg.upsample_scale
        f0 = g.f0_upsamp(f0[:, None]).transpose(1, 2)  # (1, samples, 1)
        rad = (f0 * self.harmonics).clamp(min=0) / sg.sampling_rate
        rad = rad - torch.floor(rad)
        rad = torch.cat([rad[:, :1] + phases[:, None], rad[:, 1:]], 1)
        rad = (rad[:, k // 2 - 1 :: k] + rad[:, k // 2 :: k]) / 2  # Interpolating by 1/k: the middle two of each k
        cycles = self.upsample((torch.cumsum(rad, dim=1) * k).transpose(1, 2)).transpose(1, 2)
        sines = torch.sin(2 * torch.pi * (cycles - torch.floor(cycles))) * sg.sine_amp
        uv = (f0 > sg.voiced_threshold).float()
        sines = sines * uv + (uv * sg.noise_std + (1 - uv) * sg.sine_amp / 3) * noise.transpose(1, 2)
        merged = g.m_source.l_tanh(g.m_source.l_linear(sines))
        spec, phase = g.stft.transform(merged.transpose(1, 2).squeeze(1))
        return torch.cat([spec, phase], dim=1)

To check it against `HarmonicSource`, it needs the random numbers `HarmonicSource` would draw. That turned out to be
subtler than drawing the same shapes with the same seed. Kokoro's noise comes from `torch.randn_like` on its sine
waves, which are a transposed view, and PyTorch's CPU generator fills a non-contiguous tensor with different numbers
from a contiguous one of the same shape. So the noise has to be drawn into a transposed view too:

In [ ]:
def kokoro_randoms(T, seed):
    "The initial phases and noise HarmonicSource draws for T frames after torch.manual_seed(seed), in the order it does."
    torch.manual_seed(seed)
    phases = torch.rand(1, 9)
    phases[:, 0] = 0
    noise = torch.empty(1, 9, 600 * T).transpose(1, 2).normal_().transpose(1, 2).contiguous()
    return phases, noise


def source_error(a, b):
    "Largest differences: magnitude, and phase modulo 2 pi where there's enough magnitude for a phase to mean anything."
    a, b = np.asarray(a), np.asarray(b)
    n = a.shape[1] // 2
    dp = np.abs(a[:, n:] - b[:, n:])
    return np.abs(a[:, :n] - b[:, :n]).max(), np.minimum(dp, 2 * np.pi - dp)[np.abs(b[:, :n]) > 0.01].max()


dec = exportable_decoder()
source, reference = Source(dec).eval(), HarmonicSource(dec).eval()
T = u.asr.shape[-1]
phases, noise = kokoro_randoms(T, 0)
with torch.no_grad():
    torch.manual_seed(0)
    har = reference(u.F0)
    mag, ph = source_error(source(u.F0, phases, noise), har)
print(f"magnitude within {mag:.4f} of a peak of {har[:, :11].abs().max():.2f}; phase within {ph:.3f} rad")

magnitude within 0.0013 of a peak of 1.42; phase within 0.076 rad


The same to a fraction of a percent: what's left is rounding, from adding up phase over thousands of cycles in a
different order, and it grows with the sentence. On a phone, any good Gaussian generator will do for the noise.

In [ ]:
#| export
SOURCE_IN = [
    ct.TensorType("f0", shape=(1, ct.RangeDim(8, 8192, 512))),
    ct.TensorType("phases", shape=(1, 9)),
    ct.TensorType("noise", shape=(1, 9, ct.RangeDim(2400, 2457600, 153600))),
]


def build_source():
    "Convert Source for both platforms, 32-bit, into IOS and ANDROID."
    source = Source(exportable_decoder()).eval()
    f0, phases, noise = torch.full((1, 512), 150.0), torch.zeros(1, 9), torch.randn(1, 9, 600 * 256)
    to_coreml_flexible(source, (f0, phases, noise), SOURCE_IN, ["har"], IOS / "source.mlpackage")
    to_onnx(
        source,
        (f0, phases, noise),
        ANDROID / "source.onnx",
        ["f0", "phases", "noise"],
        ["har"],
        {"f0": {1: "frames2"}, "noise": {2: "samples"}, "har": {2: "columns"}},
    )

In [ ]:
fresh("build_source()")
coreml_source = ct.models.CompiledMLModel(str(IOS / "source.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
T2 = u2.asr.shape[-1]
phases, noise = kokoro_randoms(T2, 1)
with torch.no_grad():
    torch.manual_seed(1)
    har2 = reference(u2.F0)
feed = {"f0": u2.F0.numpy(), "phases": phases.numpy(), "noise": noise.numpy()}
onnx_har = isolated(
    f"""
import onnxruntime as ort
source = ort.InferenceSession("{ANDROID / "source.onnx"}", providers=["CPUExecutionProvider"])
outputs["har"] = source.run(None, inputs)[0]
""",
    **feed,
)["har"]
for name, out in [("Core ML", predict(coreml_source, feed)["har"]), ("ONNX", onnx_har)]:
    print(
        f"{name}: magnitude within {source_error(out, har2)[0]:.4f}, phase within {source_error(out, har2)[1]:.3f} rad"
    )
print(f"{size_mb(IOS / 'source.mlpackage') * 1000:.0f} KB and {size_mb(ANDROID / 'source.onnx') * 1000:.0f} KB")

Core ML: magnitude within 0.0138, phase within 0.348 rad
ONNX: magnitude within 0.0138, phase within 0.348 rad
34 KB and 21 KB


Both runtimes agree with each other exactly. Against Kokoro the error is larger on this longer sentence, about 1% of
the peak: the same rounding in the accumulated phase, over more cycles. The decoder's own noise is far larger, as
the end-to-end scores below show. And it's a few kilobytes: the source is all arithmetic, with one tiny linear layer.

## Smaller downloads

Notebook 103 converted the decoder's body once per padded size, and notebook 104 converted it for one size. Each is a
complete copy of the same weights, so three sizes on iOS are 322 MB, and on Android, at 32 bits, 612 MB. Both runtimes
can share weights between sizes in one file: Core ML's multifunction models, and LiteRT's signatures.

In [ ]:
#| export
def shared_body(name, compress=lambda m: m):
    "notebook 103's bodies, one function per size sharing their weights, in IOS/name; `compress` applied to each first."
    desc = ct.utils.MultiFunctionDescriptor()
    parts = VOICE / "parts"
    parts.mkdir(parents=True, exist_ok=True)
    for T in BUCKETS:
        part = parts / f"{name}{T}.mlpackage"
        if not part.exists():
            compress(ct.models.MLModel(str(COREML / f"body{T}.mlpackage"), skip_model_load=True)).save(str(part))
        desc.add_function(str(part), src_function_name="main", target_function_name=f"body{T}")
    desc.default_function_name = f"body{BUCKETS[0]}"
    remove_coreml(IOS / f"{name}.mlpackage")
    ct.utils.save_multifunction(desc, str(IOS / f"{name}.mlpackage"))
    ct.utils.compile_model(str(IOS / f"{name}.mlpackage"), str(IOS / f"{name}.mlmodelc"))

In [ ]:
fresh('shared_body("body")')
fresh('shared_body("body8", lambda m: cto.linear_quantize_weights(m, LINEAR8))')
print(
    f"three sizes: {sum(size_mb(COREML / f'body{T}.mlpackage') for T in BUCKETS):.0f} MB apart, "
    f"{size_mb(IOS / 'body.mlpackage'):.0f} MB shared, {size_mb(IOS / 'body8.mlpackage'):.0f} MB shared at 8 bits"
)

three sizes: 322 MB apart, 108 MB shared, 55 MB shared at 8 bits


For LiteRT, `litert_torch` converts several modules at once as signatures of one model, and shares a weight between
them when it's the same tensor. Each size's body comes from its own `phone_decoder`, so the bodies' parameters are
tied to the first one's before converting. Then `ai_edge_quantizer` makes copies with 16-bit and 8-bit weights. It
runs in notebook 104's throwaway environment, for the same reason: `litert_torch` pins an older `typing-extensions`
than Jupyter runs with.

In [ ]:
#| export
CONVERT_SHARED = """
import sys, numpy as np, torch, litert_torch
from ai_edge_quantizer import quantizer, recipe
folder, sizes = sys.argv[1], [int(T) for T in sys.argv[2:]]
modules = [torch.export.load(f"{folder}/body{T}.pt2").module() for T in sizes]
first = dict(modules[0].named_parameters())
for m in modules[1:]:  # The same architecture: tie every parameter to the first module's, so the converter shares it
    for name, p in list(m.named_parameters()):
        assert torch.equal(p, first[name]), name
        owner, _, leaf = name.rpartition(".")
        setattr(m.get_submodule(owner) if owner else m, leaf, first[name])
converter = litert_torch
for T, m in zip(sizes, modules):
    io = np.load(f"{folder}/body{T}.npz")
    converter = converter.signature(f"body{T}", m, tuple(torch.tensor(io[f"arr_{i}"]) for i in range(5)))
converter.convert().export(f"{folder}/body32.tflite")
for name in ["fp16", "int8"]:
    q = quantizer.Quantizer(f"{folder}/body32.tflite")
    if name == "fp16":
        q.add_weight_only_config(regex=".*", operation_name="*", num_bits=16, algorithm_key="float_casting")
    else:
        q.load_quantization_recipe(recipe.weight_only_wi8_afp32())
    q.quantize().export_model(f"{folder}/body_{name}.tflite", overwrite=True)
"""


def shared_tflite(u):
    "Notebook 104's body for every size in one LiteRT model, at 32, 16 and 8 bits, in ANDROID. Inputs from `u`."
    for T in BUCKETS:
        source, body = phone_decoder(T)
        with torch.no_grad(), warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="No complete tensor")  # Saving tensors that share storage
            f0 = tile(u.F0, 2 * T)
            args = (tile(u.asr, T), f0, tile(u.N, 2 * T), u.style, source(f0))
            torch.export.save(torch.export.export(body, args), ANDROID / f"body{T}.pt2")
            np.savez(ANDROID / f"body{T}.npz", *[a.numpy() for a in args], out=body(*args).numpy())
    uv = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
    env = [uv, "run", "-q", "--no-project", "--python", "3.13", "--with", "torch==2.14.0", "--with", "litert-torch"]
    subprocess.run(
        [*env, "python", "-c", CONVERT_SHARED, str(ANDROID), *map(str, BUCKETS)],
        check=True,
        capture_output=True,
        text=True,
    )
    for T in BUCKETS:
        (ANDROID / f"body{T}.pt2").unlink()

In [ ]:
shared_tflite(u)
for name in ["body32", "body_fp16", "body_int8"]:
    print(f"{name}.tflite: {size_mb(ANDROID / f'{name}.tflite'):.0f} MB")

body32.tflite: 220 MB
body_fp16.tflite: 114 MB
body_int8.tflite: 63 MB


Down from 612 MB to 220 at 32 bits, 114 at 16 and 63 at 8. Checked against PyTorch on the Mac's CPU, where LiteRT runs
without a GPU, by spectral distance in the bar's units:

In [ ]:
#| export
LITERT_BODY = """
from ai_edge_litert.interpreter import Interpreter
runners = {}
def body(path, T, *args):
    if path not in runners:
        it = Interpreter(model_path=path)
        runners[path] = (it, {B: it.get_signature_runner(f"body{B}") for B in BUCKETS})
    r = runners[path][1][T]
    names = sorted(r.get_input_details(), key=lambda n: int(n.rsplit("_", 1)[-1]) if n[-1].isdigit() else 0)
    return list(r(**dict(zip(names, args))).values())[0]
"""

In [ ]:
outs = isolated(f"""
BUCKETS = {BUCKETS}
{LITERT_BODY}
for name in ["body32", "body_fp16", "body_int8"]:
    for T in BUCKETS:
        io = np.load("{ANDROID}/body%d.npz" % T)
        outputs[f"{{name}}_{{T}}"] = body("{ANDROID}/%s.tflite" % name, T, *[io[f"arr_{{i}}"] for i in range(5)])
""")
real = u.asr.shape[-1] * FRAME  # The sentence, before the padding
for name in ["body32", "body_fp16", "body_int8"]:
    d = [
        distance(outs[f"{name}_{T}"].reshape(-1)[:real], np.load(ANDROID / f"body{T}.npz")["out"].reshape(-1)[:real])
        for T in BUCKETS
    ]
    print(f"{name}: {np.mean(d) / floor:.2f}x the seed floor from PyTorch")

body32: 0.00x the seed floor from PyTorch
body_fp16: 0.02x the seed floor from PyTorch
body_int8: 1.27x the seed floor from PyTorch


16 bits costs nothing; 8 bits costs about as much as a different seed. Whether 8-bit weights would also run on the
Pixel's GPU, and how they sound, needs the phone.

The voices themselves are small: each is a table of 510 style vectors, one per sentence length in phonemes, 0.5 MB
as 32-bit floats.

In [ ]:
#| export
def export_voice(voice, folder):
    "A voice's style table as raw little-endian float32, (510, 256), in folder/voice.bin."
    table = pipeline(voice).load_voice(voice).squeeze(1).numpy().astype("<f4")
    folder.mkdir(parents=True, exist_ok=True)
    table.tofile(folder / f"{voice}.bin")
    return table.shape

In [ ]:
for voice in ["af_heart", "bf_emma"]:
    for folder in [IOS / "voices", ANDROID / "voices"]:
        shape = export_voice(voice, folder)
print(shape, f"{size_mb(IOS / 'voices' / 'af_heart.bin'):.2f} MB each")

(510, 256) 0.52 MB each


## All together

`synthesize` is what an app does, written in numpy so it's plain what each step is: run `Tokens`, turn durations into
frames, run `Prosody`, pick the smallest size that fits, pad as notebook 103 does, draw the source's random numbers,
run the source and the body, and trim. Each platform supplies four functions.

In [ ]:
#| export
def synthesize(ids, style, tokens, prosody, source, body, rng):
    "Audio for phoneme ids (1, n) and a style (1, 256), through four runtime functions. Longest: BUCKETS[-1] frames."
    d, t_en, duration = tokens(ids, style)
    f = frames(duration)
    F0, N = prosody(per_frame(d.transpose(0, 2, 1), f), style)
    T = int(f.sum())
    B = next(b for b in BUCKETS if b >= T)  # The app splits longer sentences before they get here
    f0 = repeat_to(F0, 2 * B).astype(np.float32)
    phases = rng.random((1, 9), dtype=np.float32)
    phases[0, 0] = 0
    har = source(f0, phases, rng.standard_normal((1, 9, 600 * B), dtype=np.float32))
    asr, n = repeat_to(per_frame(t_en, f), B).astype(np.float32), repeat_to(N, 2 * B).astype(np.float32)
    return np.asarray(body(B, asr, f0, n, style[:, :128], har)).reshape(-1)[: T * FRAME]


def ios_parts(body="body", front=""):
    "The four functions on Core ML: the front half and source on the CPU, the body on the GPU."
    cpu = lambda name: ct.models.CompiledMLModel(str(IOS / f"{name}.mlmodelc"), ct.ComputeUnit.CPU_ONLY)
    tok, pro, src = cpu(f"tokens{front}"), cpu(f"prosody{front}"), cpu("source")
    bodies = {
        T: ct.models.CompiledMLModel(
            str(IOS / f"{body}.mlmodelc"), ct.ComputeUnit.CPU_AND_GPU, function_name=f"body{T}"
        )
        for T in BUCKETS
    }

    def tokens(ids, s):
        o = predict(tok, {"ids": ids.astype(np.int32), "style": s})
        return o["d"], o["t_en"], o["duration"]

    def prosody(en, s):
        o = predict(pro, {"en": en, "style": s})
        return o["F0"], o["N"]

    return {
        "tokens": tokens,
        "prosody": prosody,
        "source": lambda f0, p, n: predict(src, {"f0": f0, "phases": p, "noise": n})["har"],
        "body": lambda B, asr, f0, n, s, har: predict(
            bodies[B], {"asr": asr, "f0": f0, "n": n, "style": s, "har": har}
        )["out"],
    }

Scored against Kokoro, with the 16-bit body and with the 8-bit body and front half:

In [ ]:
inputs = [(phoneme_ids(u.phonemes), voice_style(u.voice, u.phonemes)) for u in utterances]


def score(name, parts):
    rng = np.random.default_rng(0)
    synthesize(*inputs[0], **parts, rng=rng)
    t = time.perf_counter()
    outs = [synthesize(*x, **parts, rng=rng) for x in inputs]
    rtf = seconds / (time.perf_counter() - t)
    report(name, outs, rtf)


def report(name, outs, rtf):
    lengths = sum(len(o) == len(r) for o, r in zip(outs, ref))
    d = np.mean([distance(o, r) for o, r in zip(outs, ref)]) / floor
    a = np.mean([aligned(o, r) for o, r in zip(outs, ref)]) / aligned_floor
    print(f"{name}: {rtf:.1f}x real time, {lengths}/10 Kokoro's length, {d:.2f}x the seed floor, {a:.2f}x aligned")


for body, front, name in [
    ("body", "", "16-bit body"),
    ("body8", "", "8-bit body"),
    ("body", "8", "8-bit front half"),
    ("body8", "8", "8-bit body and front half"),
]:
    score(f"iOS, {name}", ios_parts(body, front))

iOS, 16-bit body: 28.6x real time, 10/10 Kokoro's length, 3.91x the seed floor, 3.90x aligned


iOS, 8-bit body: 10.7x real time, 10/10 Kokoro's length, 3.93x the seed floor, 3.91x aligned


iOS, 8-bit front half: 29.5x real time, 4/10 Kokoro's length, 11.79x the seed floor, 4.65x aligned


iOS, 8-bit body and front half: 10.6x real time, 4/10 Kokoro's length, 11.80x the seed floor, 4.66x aligned


The 16-bit pipeline is where notebook 103's decoder was, and the same length as Kokoro's every time: nothing is lost
by moving everything else off PyTorch. The 8-bit body costs nothing either, though it's a third of the speed on the
Mac's GPU, as appendix 193 found: still several times what playback at 3x needs, but the phones will have to say. The
8-bit front half is different. Its durations move by a frame here and there, so most sentences come out a different
length, and even aligned in time it's further from Kokoro than another seed is. It stays at 32 bits.

Android's runtimes run in a process of their own: ONNX Runtime and LiteRT in the same process as Core ML have crashed
each other here. The script imports this module, so it's the same `synthesize`:

In [ ]:
#| export
ANDROID_PARTS = """
import time, onnxruntime as ort
from text_to_speech.phone import ANDROID, BUCKETS, synthesize
sess = lambda name: ort.InferenceSession(str(ANDROID / f"{name}.onnx"), providers=["CPUExecutionProvider"])
tok, pro, src = sess("tokens"), sess("prosody"), sess("source")
parts = dict(tokens=lambda i, s: tuple(tok.run(None, {"ids": i.astype(np.int64), "style": s})),
             prosody=lambda en, s: tuple(pro.run(None, {"en": en, "style": s})),
             source=lambda f0, p, n: src.run(None, {"f0": f0, "phases": p, "noise": n})[0],
             body=lambda T, *args: body(str(ANDROID / BODY), T, *args))
"""


def android_check(inputs, body="body_fp16.tflite"):
    "Run `synthesize` on ONNX Runtime and LiteRT, isolated: (the audio for each (ids, style), seconds taken)."
    arrays = {f"{k}{i}": a for i, (ids, style) in enumerate(inputs) for k, a in [("ids", ids), ("style", style)]}
    out = isolated(
        LITERT_BODY
        + ANDROID_PARTS.replace("BODY", repr(body))
        + f"""
rng = np.random.default_rng(0)
t = time.perf_counter()
for i in range({len(inputs)}):
    outputs[f"audio{{i}}"] = synthesize(inputs[f"ids{{i}}"], inputs[f"style{{i}}"], **parts, rng=rng)
outputs["seconds"] = np.array(time.perf_counter() - t)
""",
        **arrays,
    )
    return [out[f"audio{i}"] for i in range(len(inputs))], float(out["seconds"])

In [ ]:
for body in ["body_fp16.tflite", "body_int8.tflite"]:
    outs, elapsed = android_check(inputs, body)
    report(f"Android, {body} (the Mac's CPU)", outs, seconds / elapsed)

Android, body_fp16.tflite (the Mac's CPU): 1.1x real time, 10/10 Kokoro's length, 3.92x the seed floor, 3.90x aligned


Android, body_int8.tflite (the Mac's CPU): 1.1x real time, 10/10 Kokoro's length, 4.57x the seed floor, 4.56x aligned


The 16-bit body matches Kokoro as closely as Core ML does. On the Mac's CPU, LiteRT only just keeps up; on the
Pixel's GPU, notebook 104 measured the body at about 10x real time. The 8-bit body loses a little more, as it did
alone.

## What an app downloads

In [ ]:
ios = {"tokens.mlmodelc", "prosody.mlmodelc", "source.mlmodelc", "body8.mlmodelc", "voices"}
android = {"tokens.onnx", "prosody.onnx", "source.onnx", "body_fp16.tflite", "voices"}
for name, folder, files in [("iOS", IOS, ios), ("Android", ANDROID, android)]:
    print(
        f"{name}: {sum(size_mb(folder / f) for f in files):.0f} MB:",
        ", ".join(f"{f} {size_mb(folder / f):.1f}" for f in sorted(files)),
    )

iOS: 167 MB: body8.mlmodelc 54.6, prosody.mlmodelc 33.7, source.mlmodelc 0.0, tokens.mlmodelc 78.1, voices 1.0
Android: 227 MB: body_fp16.tflite 114.1, prosody.onnx 33.7, source.onnx 0.0, tokens.onnx 78.3, voices 1.0


Plus the next notebook's text-to-phoneme files, which both platforms share. The front half is now as big as the body.
Shrinking it without moving durations, 16-bit weights computed in 32 bits, say, is the obvious next step.

## Summary

- **The front half** is two modules, `Tokens` per phoneme and `Prosody` per frame, with two lines of app code between
  them. They reproduce `analyse` exactly.
- **Any length**: the LSTMs need each sentence's own length, which `torch.export` can't give them. TorchScript into
  Core ML and the ONNX exporter keep the LSTMs whole, at any length, and both match PyTorch. They must compute in 32
  bits (ALBERT overflows at 16), and 8-bit weights move the durations enough to hear.
- **The source** takes its randomness as inputs and converts for both. Reproducing Kokoro's noise exactly took
  drawing it into a transposed view.
- **Smaller downloads**: one file per platform holds the body at every size with its weights shared. iOS: 322 MB to
  108 at 16 bits, 55 at 8, which costs nothing. Android: 612 MB to 114 at 16 bits; 8 bits costs about a seed.
- **End to end**, through each platform's runtimes and only an app's glue, the voice is as close to Kokoro as
  notebook 103's decoder was. iOS downloads about 170 MB, Android about 230.
- **Two crashes**, both in the Mac's Python bindings rather than on a phone: Core ML frees prediction inputs on a
  thread of its own, and ONNX Runtime and LiteRT can't share a process with it.

Still to check on the phones: speed, the 8-bit body on the iPhone's GPU, and listening.